# Week 5 - Spark Data Cleaning, Transformation and Aggregation using DataFrames

**Celebal Technologies Summer Internship 2026**

**Name:** Aditya Kumar

**Week:** 5

**Technology:** Apache Spark (PySpark)

## Objective

The objective of this assignment is to understand Apache Spark fundamentals and perform data cleaning, transformation, filtering, aggregation, schema modification, and build a complete data processing pipeline using Spark DataFrames.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Spark Session Initialization

In [2]:
spark = SparkSession.builder \
    .appName("Celebal Week 5") \
    .getOrCreate()

print("Spark Session Created Successfully!")

Spark Session Created Successfully!


## Loading the Dataset

In [3]:
df = spark.read.csv(
    "ecommerce_sales_data.csv",
    header=True,
    inferSchema=True
)

## Dataset Preview

In [4]:
df.show(5)

+----------+--------+-----------+----------+----------------+------------+-------------+----------+------------+----------------+-----------------+------------+-------------+-------------+------------+---------------+----------------+-------------+-----------+----------------+
|      Date|Order ID|Customer ID|Product ID|Product Category|Product Name|Quantity Sold|Unit Price|Discount (%)|  Payment Method|Customer Location|Order Status|Shipping Cost|Profit Margin|Customer Age|Customer Gender|Customer Segment|Review Rating|Total Sales|Discounted Price|
+----------+--------+-----------+----------+----------------+------------+-------------+----------+------------+----------------+-----------------+------------+-------------+-------------+------------+---------------+----------------+-------------+-----------+----------------+
|2024-01-01|ORD00001|   CUST1654|    PROD23|        Clothing|     T-Shirt|            5|     28.75|          15|Cash on Delivery|      Los Angeles|   Completed|      

## Dataset Schema

In [5]:
df.printSchema()

root
 |-- Date: date (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Product Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Quantity Sold: integer (nullable = true)
 |-- Unit Price: double (nullable = true)
 |-- Discount (%): integer (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Customer Location: string (nullable = true)
 |-- Order Status: string (nullable = true)
 |-- Shipping Cost: double (nullable = true)
 |-- Profit Margin: double (nullable = true)
 |-- Customer Age: integer (nullable = true)
 |-- Customer Gender: string (nullable = true)
 |-- Customer Segment: string (nullable = true)
 |-- Review Rating: integer (nullable = true)
 |-- Total Sales: double (nullable = true)
 |-- Discounted Price: double (nullable = true)



## Dataset Information

In [6]:
print("Total Records:", df.count())
print("Total Columns:", len(df.columns))

Total Records: 100
Total Columns: 20


In [7]:
df.columns

['Date',
 'Order ID',
 'Customer ID',
 'Product ID',
 'Product Category',
 'Product Name',
 'Quantity Sold',
 'Unit Price',
 'Discount (%)',
 'Payment Method',
 'Customer Location',
 'Order Status',
 'Shipping Cost',
 'Profit Margin',
 'Customer Age',
 'Customer Gender',
 'Customer Segment',
 'Review Rating',
 'Total Sales',
 'Discounted Price']

# Q1. What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

### Answer

Traditional MapReduce processes data by writing intermediate results to disk after every stage. This increases execution time and makes it inefficient for iterative and real-time workloads.

Apache Spark addresses these limitations by performing computations in memory (RAM), significantly reducing disk I/O and improving execution speed. Spark also provides a unified framework that supports SQL, machine learning, graph processing, and streaming, making it more efficient and easier to use for modern big data applications.

# Q2. Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

### Answer

Spark stores intermediate data in memory instead of writing it to disk after every operation. Since iterative machine learning algorithms repeatedly process the same dataset, keeping the data in memory avoids repeated disk reads and writes.

This significantly reduces latency, improves execution speed, and makes Spark much faster than traditional disk-based systems such as MapReduce.

# Q3. Remove duplicate rows based on user_id and transaction_date.

### Note

The provided dataset does not contain the columns **user_id** and **transaction_date**. Therefore, the equivalent columns **Customer ID** and **Date** are used.

In [8]:
duplicate_removed_df = df.dropDuplicates(["Customer ID", "Date"])

print("Original Records :", df.count())
print("Records After Removing Duplicates :", duplicate_removed_df.count())

duplicate_removed_df.show(5)

Original Records : 100
Records After Removing Duplicates : 100
+----------+--------+-----------+----------+----------------+------------+-------------+----------+------------+--------------+-----------------+------------+-------------+-------------+------------+---------------+----------------+-------------+-----------+----------------+
|      Date|Order ID|Customer ID|Product ID|Product Category|Product Name|Quantity Sold|Unit Price|Discount (%)|Payment Method|Customer Location|Order Status|Shipping Cost|Profit Margin|Customer Age|Customer Gender|Customer Segment|Review Rating|Total Sales|Discounted Price|
+----------+--------+-----------+----------+----------------+------------+-------------+----------+------------+--------------+-----------------+------------+-------------+-------------+------------+---------------+----------------+-------------+-----------+----------------+
|2024-02-08|ORD00039|   CUST1006|    PROD42|        Clothing|Coffee Maker|            2|    112.99|          

# Q4. Filter rows where the region is 'West' and calculate the average sale amount grouped by product category.

### Note

The dataset does not contain a **Region** column. The **Customer Location** column is used instead.

For demonstration purposes, **Los Angeles** is treated as a representative location for the West region.

Similarly,

- Product Category → product_category
- Total Sales → sale_amount

In [9]:
from pyspark.sql.functions import avg

filtered_sales = (
    df.filter(col("Customer Location") == "Los Angeles")
      .groupBy("Product Category")
      .agg(avg("Total Sales").alias("Average Sales"))
)

filtered_sales.show()

+----------------+-----------------+
|Product Category|    Average Sales|
+----------------+-----------------+
|  Home & Kitchen|           929.61|
|          Sports|          1595.72|
|     Electronics|992.9599999999999|
|        Clothing|308.0028571428571|
|          Beauty|          1135.48|
+----------------+-----------------+



# Q5. What is the difference between `.na.drop()` and `.na.fill()`? Provide a code example of filling null values in a status column with the string `"Unknown"`.

### Answer

- **`.na.drop()`** removes rows that contain null values.
- **`.na.fill()`** replaces null values with a specified value while keeping the rows intact.

In this dataset, the **Order Status** column is used as the status column for demonstration.

In [10]:
# Create a sample null value for demonstration
demo_df = df.withColumn(
    "Order Status",
    when(col("Order ID") == "ORD00005", None)
    .otherwise(col("Order Status"))
)

filled_df = demo_df.na.fill({"Order Status": "Unknown"})

filled_df.select("Order ID", "Order Status").show(10)

+--------+------------+
|Order ID|Order Status|
+--------+------------+
|ORD00001|   Completed|
|ORD00002|   Cancelled|
|ORD00003|    Returned|
|ORD00004|   Cancelled|
|ORD00005|     Unknown|
|ORD00006|   Cancelled|
|ORD00007|   Cancelled|
|ORD00008|    Returned|
|ORD00009|    Returned|
|ORD00010|   Cancelled|
+--------+------------+
only showing top 10 rows


# Q6. Find the total count of records for each city, but only for cities where the count is greater than 100.

### Note

The provided dataset contains only 100 records, so no city can satisfy the condition **count > 100**.

For demonstration purposes, the condition has been modified to **count > 20**.

In [11]:
city_count = (
    df.groupBy("Customer Location")
      .count()
      .filter(col("count") > 20)
)

city_count.show()

+-----------------+-----+
|Customer Location|count|
+-----------------+-----+
|      Los Angeles|   22|
|         New York|   22|
+-----------------+-----+



# Q7. How does the immutability of Spark DataFrames affect data cleaning operations such as dropping or renaming columns?

### Answer

Spark DataFrames are immutable, meaning they cannot be modified after creation.

Operations such as dropping columns, renaming columns, filtering rows, or updating values do not change the original DataFrame. Instead, Spark creates a new DataFrame containing the requested modifications.

This approach improves fault tolerance, enables query optimization, and preserves the original data throughout the processing pipeline.

In [12]:
new_df = df.drop("Discount (%)")

renamed_df = df.withColumnRenamed("Customer Age", "Age")

print("Original Columns:")
print(df.columns)

print("\nRenamed DataFrame Columns:")
print(renamed_df.columns)

Original Columns:
['Date', 'Order ID', 'Customer ID', 'Product ID', 'Product Category', 'Product Name', 'Quantity Sold', 'Unit Price', 'Discount (%)', 'Payment Method', 'Customer Location', 'Order Status', 'Shipping Cost', 'Profit Margin', 'Customer Age', 'Customer Gender', 'Customer Segment', 'Review Rating', 'Total Sales', 'Discounted Price']

Renamed DataFrame Columns:
['Date', 'Order ID', 'Customer ID', 'Product ID', 'Product Category', 'Product Name', 'Quantity Sold', 'Unit Price', 'Discount (%)', 'Payment Method', 'Customer Location', 'Order Status', 'Shipping Cost', 'Profit Margin', 'Age', 'Customer Gender', 'Customer Segment', 'Review Rating', 'Total Sales', 'Discounted Price']


# Q8. Filter rows where the age is between 18 and 30 (inclusive) and the subscription is "Premium".

### Note

The dataset contains a **Customer Age** column but does not include a **Subscription** column.

A temporary Subscription column is created for demonstration purposes.

In [13]:
subscription_df = df.withColumn(
    "Subscription",
    when(col("Customer Age").between(18, 30), "Premium")
    .otherwise("Standard")
)

filtered_df = subscription_df.filter(
    (col("Customer Age").between(18, 30)) &
    (col("Subscription") == "Premium")
)

filtered_df.select(
    "Customer ID",
    "Customer Age",
    "Subscription",
    "Product Category",
    "Total Sales"
).show()

+-----------+------------+------------+----------------+-----------+
|Customer ID|Customer Age|Subscription|Product Category|Total Sales|
+-----------+------------+------------+----------------+-----------+
|   CUST1025|          25|     Premium|     Electronics|     896.95|
|   CUST1759|          26|     Premium|  Home & Kitchen|    1186.05|
|   CUST1281|          20|     Premium|          Sports|     223.83|
|   CUST1250|          20|     Premium|          Sports|     984.56|
|   CUST1754|          25|     Premium|  Home & Kitchen|     103.14|
|   CUST1104|          24|     Premium|        Clothing|     158.75|
|   CUST1913|          26|     Premium|     Electronics|    1215.57|
|   CUST1223|          27|     Premium|        Clothing|     946.05|
|   CUST1616|          24|     Premium|          Sports|    1804.08|
|   CUST1665|          20|     Premium|          Sports|     563.32|
|   CUST1284|          24|     Premium|          Sports|     208.98|
|   CUST1825|          21|     Pre

# Q9. When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like `sum()` or `avg()`?

### Answer

Handling null values before performing mathematical aggregations is important because null values can affect the accuracy and reliability of the results.

Replacing or removing null values ensures that aggregation functions such as `sum()`, `avg()`, `min()`, and `max()` produce meaningful and consistent results while preventing unexpected outcomes during data analysis.

In [14]:
# Create a sample null value for demonstration
sales_df = df.withColumn(
    "Total Sales",
    when(col("Order ID") == "ORD00008", None)
    .otherwise(col("Total Sales"))
)

clean_sales_df = sales_df.na.fill({"Total Sales": 0})

clean_sales_df.groupBy("Product Category") \
    .sum("Total Sales") \
    .show()

+----------------+------------------+
|Product Category|  sum(Total Sales)|
+----------------+------------------+
|  Home & Kitchen|13858.380000000001|
|          Sports|19059.489999999998|
|     Electronics|          15271.19|
|        Clothing|          13681.89|
|          Beauty|14609.570000000002|
+----------------+------------------+



# Q10. Revise a column named `raw_timestamp` by casting it to `TimestampType` and renaming it to `event_time`.

### Note

The dataset does not contain a `raw_timestamp` column. Therefore, the existing **Date** column is used as an equivalent timestamp field.

In [15]:
from pyspark.sql.functions import to_timestamp

timestamp_df = df.withColumn(
    "event_time",
    to_timestamp("Date", "yyyy-MM-dd")
).drop("Date")

timestamp_df.select("event_time").show(10)

timestamp_df.printSchema()

+-------------------+
|         event_time|
+-------------------+
|2024-01-01 00:00:00|
|2024-01-02 00:00:00|
|2024-01-03 00:00:00|
|2024-01-04 00:00:00|
|2024-01-05 00:00:00|
|2024-01-06 00:00:00|
|2024-01-07 00:00:00|
|2024-01-08 00:00:00|
|2024-01-09 00:00:00|
|2024-01-10 00:00:00|
+-------------------+
only showing top 10 rows
root
 |-- Order ID: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Product Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Quantity Sold: integer (nullable = true)
 |-- Unit Price: double (nullable = true)
 |-- Discount (%): integer (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Customer Location: string (nullable = true)
 |-- Order Status: string (nullable = true)
 |-- Shipping Cost: double (nullable = true)
 |-- Profit Margin: double (nullable = true)
 |-- Customer Age: integer (nullable = true)
 |-- Customer Gender: string (nullable

# Q11. Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

### Answer

A shuffle is the process of redistributing data across different partitions in a Spark cluster.

Operations such as `groupBy()`, `join()`, and `reduceByKey()` require records with the same key to be brought together before processing.

Since data must move between partitions, shuffle is considered a **wide transformation**. It involves network communication and disk I/O, making it one of the most expensive operations in Spark.

In [16]:
df.groupBy("Product Category") \
    .sum("Total Sales") \
    .show()

+----------------+------------------+
|Product Category|  sum(Total Sales)|
+----------------+------------------+
|  Home & Kitchen|13858.380000000001|
|          Sports|19059.489999999998|
|     Electronics|          15271.19|
|        Clothing|14132.649999999998|
|          Beauty|14609.570000000002|
+----------------+------------------+



# Q12. Remove rows where the email column contains null values OR the username is an empty string.

### Note

The dataset does not contain **Email** and **Username** columns.

Temporary columns are created for demonstration purposes.

In [17]:
user_df = df.withColumn(
    "Email",
    lit("customer@example.com")
).withColumn(
    "Username",
    lit("customer")
)

user_df = user_df.withColumn(
    "Email",
    when(col("Customer ID") == "CUST001", None)
    .otherwise(col("Email"))
)

user_df = user_df.withColumn(
    "Username",
    when(col("Customer ID") == "CUST002", "")
    .otherwise(col("Username"))
)

clean_user_df = user_df.filter(
    col("Email").isNotNull() &
    (col("Username") != "")
)

clean_user_df.select(
    "Customer ID",
    "Email",
    "Username"
).show()

+-----------+--------------------+--------+
|Customer ID|               Email|Username|
+-----------+--------------------+--------+
|   CUST1654|customer@example.com|customer|
|   CUST1114|customer@example.com|customer|
|   CUST1025|customer@example.com|customer|
|   CUST1759|customer@example.com|customer|
|   CUST1281|customer@example.com|customer|
|   CUST1250|customer@example.com|customer|
|   CUST1228|customer@example.com|customer|
|   CUST1142|customer@example.com|customer|
|   CUST1754|customer@example.com|customer|
|   CUST1104|customer@example.com|customer|
|   CUST1692|customer@example.com|customer|
|   CUST1758|customer@example.com|customer|
|   CUST1913|customer@example.com|customer|
|   CUST1558|customer@example.com|customer|
|   CUST1089|customer@example.com|customer|
|   CUST1604|customer@example.com|customer|
|   CUST1432|customer@example.com|customer|
|   CUST1032|customer@example.com|customer|
|   CUST1030|customer@example.com|customer|
|   CUST1095|customer@example.co

# Q13. How do you use the `.agg()` function to calculate multiple statistics at once, such as the minimum, maximum, and average of the `price` column?

### Answer

The `.agg()` function allows multiple aggregate functions to be applied in a single operation. It improves readability and efficiency by calculating multiple statistics simultaneously.

In this dataset, the **Unit Price** column is used as the equivalent of the **price** column.

In [18]:
df.agg(
    min("Unit Price").alias("Minimum Price"),
    max("Unit Price").alias("Maximum Price"),
    avg("Unit Price").alias("Average Price")
).show()

+-------------+-------------+------------------+
|Minimum Price|Maximum Price|     Average Price|
+-------------+-------------+------------------+
|        28.75|       492.28|283.79790000000014|
+-------------+-------------+------------------+



# Q14. In the context of cleaning a dataset, what is the risk of using `inferSchema=True` when the source data contains messy or inconsistent date formats?

### Answer

When `inferSchema=True` is enabled, Spark automatically determines the data type of each column.

If a dataset contains inconsistent or messy date formats, Spark may infer incorrect data types or convert some values to `null`. This can lead to incorrect analysis, failed transformations, and data quality issues.

For production environments, it is generally recommended to define the schema explicitly instead of relying entirely on schema inference.

In [19]:
df.printSchema()

root
 |-- Date: date (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Product Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Quantity Sold: integer (nullable = true)
 |-- Unit Price: double (nullable = true)
 |-- Discount (%): integer (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Customer Location: string (nullable = true)
 |-- Order Status: string (nullable = true)
 |-- Shipping Cost: double (nullable = true)
 |-- Profit Margin: double (nullable = true)
 |-- Customer Age: integer (nullable = true)
 |-- Customer Gender: string (nullable = true)
 |-- Customer Segment: string (nullable = true)
 |-- Review Rating: integer (nullable = true)
 |-- Total Sales: double (nullable = true)
 |-- Discounted Price: double (nullable = true)



# Q15. Build a final processing pipeline that:

1. Filters out duplicate records.
2. Fills null prices with 0.
3. Groups by `store_id` to calculate total revenue.

### Note

The dataset does not contain a **store_id** column.

For demonstration purposes, **Product Category** is used as the grouping column.

In [20]:
pipeline_df = df.dropDuplicates()

pipeline_df = pipeline_df.na.fill({
    "Unit Price": 0
})

result_df = pipeline_df.groupBy("Product Category") \
    .agg(
        sum("Total Sales").alias("Total Revenue")
    )

result_df.show()

+----------------+------------------+
|Product Category|     Total Revenue|
+----------------+------------------+
|  Home & Kitchen|          13858.38|
|          Sports|19059.489999999998|
|     Electronics|          15271.19|
|        Clothing|          14132.65|
|          Beauty|14609.570000000002|
+----------------+------------------+



# Conclusion

In this assignment, Apache Spark DataFrames were used to perform various data processing operations, including data cleaning, filtering, aggregation, schema modification, and grouping.

The implementation demonstrated important Spark concepts such as in-memory computing, DataFrame immutability, shuffle operations, and wide transformations. Practical exercises included handling duplicate records, managing null values, filtering data, performing aggregations, modifying schemas, and constructing a complete data processing pipeline.

Overall, this assignment provided hands-on experience with PySpark DataFrames and strengthened the understanding of scalable data processing techniques used in modern big data applications.

---